In [19]:
%pip install oauth2client

import pandas as pd
import numpy as np
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import matplotlib.pyplot as plt

# ใช้ username และ password แทน service account json
from gspread.exceptions import APIError

import os

# Define the scope for Google Sheets and Drive API
scope = ['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive']

credentials_path = r"..\Service Account\gen-lang-client-0406795509-bc06ec2e33dc.json"
# Use ServiceAccountCredentials and gspread.authorize only if credentials file exists
if os.path.exists(credentials_path):
	credentials = ServiceAccountCredentials.from_json_keyfile_name(credentials_path, scope)
	gc = gspread.authorize(credentials)

	# ── Master (Store Master) from Google Sheets ──────────────────────
	sheet_master = gc.open_by_url("https://docs.google.com/spreadsheets/d/1PiFiOJyxoI6aDR9xdU4HIl8KoDG-50PwYuL4Bm9f8Fk/edit?gid=0#gid=0")
	ws_master = sheet_master.worksheet("Store master")
	Master = pd.DataFrame(ws_master.get_all_records())

	# ── Ref_Survey_Database ─────────────────────────────────────────────
	sheet = gc.open_by_url("https://docs.google.com/spreadsheets/d/1jNP96tfKsehXOS1EknpoO5fkcDbonyu4mEWxxxa3zKE/edit?usp=sharing")
	worksheet = sheet.worksheet("Ref_Survey_Database")
	data = worksheet.get_all_records()
	df_refig_survey = pd.DataFrame(data)
	
	# ── Aircon_Survey_Database ─────────────────────────────────────────────
	sheet = gc.open_by_url("https://docs.google.com/spreadsheets/d/1jNP96tfKsehXOS1EknpoO5fkcDbonyu4mEWxxxa3zKE/edit?usp=sharing")
	worksheet = sheet.worksheet("Aircon_Survey_Database")
	data = worksheet.get_all_records()
	df_aircon_survey = pd.DataFrame(data)


else:
	print(f"Warning: {credentials_path} not found. Skipping Google Sheets import.")

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [20]:

# ใช้ข้อมูลล่าสุดของแต่ละ Branch Code (ทั้ง Refig และ Aircon)
df_refig_survey_Clean = (
    df_refig_survey
    .dropna(subset=['Branch Code'])
    .assign(Timestamp=pd.to_datetime(df_refig_survey['Timestamp'], errors='coerce'))
    .sort_values(['Branch Code', 'Timestamp'], na_position='first')
    .drop_duplicates(subset=['Branch Code'], keep='last')
    .reset_index(drop=True)
)

df_aircon_survey_clean = (
    df_aircon_survey
    .dropna(subset=['Branch Code'])
    .assign(Timestamp=pd.to_datetime(df_aircon_survey['Timestamp'], errors='coerce'))
    .sort_values(['Branch Code', 'Timestamp'], na_position='first')
    .drop_duplicates(subset=['Branch Code'], keep='last')
    .reset_index(drop=True)
)

In [21]:
merged_critical_df = pd.merge(df_refig_survey_Clean, df_aircon_survey_clean, on='Branch Code', how='outer', suffixes=('_Refig', '_Aircon'))

In [22]:
cols = df_refig_survey_Clean.columns.tolist()
qty_cols = [c for c in cols if c.endswith('Qty')]
print("=== จำนวนตู้แต่ละประเภท (Clean data) ===")
for c in qty_cols:
    total = pd.to_numeric(df_refig_survey_Clean[c], errors='coerce').sum()
    stores = (pd.to_numeric(df_refig_survey_Clean[c], errors='coerce') > 0).sum()
    print(f"{c:30s}  รวม {int(total):5d} ตัว  ({stores} สาขา)")

=== จำนวนตู้แต่ละประเภท (Clean data) ===
Ref Open Qty                    รวม  6451 ตัว  (1180 สาขา)
Ref Bev Walk In Qty             รวม   332 ตัว  (193 สาขา)
Ref Frozen3 Door Qty            รวม    91 ตัว  (142 สาขา)
Ref Ice Cream Qty               รวม  2095 ตัว  (1098 สาขา)
Ref Ice Qty                     รวม  1794 ตัว  (1112 สาขา)
Ref Frozen1 Door Qty            รวม  2855 ตัว  (999 สาขา)
Ref Bev Plugin Qty              รวม  1882 ตัว  (741 สาขา)
Ref Frozen2 Door Qty            รวม   129 ตัว  (132 สาขา)
Ref Bev Remote Qty              รวม   620 ตัว  (266 สาขา)
Ref Ss4 Door Qty                รวม  1314 ตัว  (708 สาขา)


In [23]:
cols = df_refig_survey_Clean.columns.tolist()
type_prefixes = {
    'Open':       'Ref Open',
    'Bev Walk In':'Ref Bev Walk In',
    'Frozen 3D':  'Ref Frozen3 Door',
    'Ice Cream':  'Ref Ice Cream',
    'Ice':        'Ref Ice',
    'Frozen 1D':  'Ref Frozen1 Door',
    'Bev Plugin': 'Ref Bev Plugin',
    'Frozen 2D':  'Ref Frozen2 Door',
    'Bev Remote': 'Ref Bev Remote',
    'SS 4D':      'Ref Ss4 Door',
}

print("=== ยี่ห้อแต่ละประเภท ===")
for label, prefix in type_prefixes.items():
    brand_cols = [c for c in cols if c.startswith(prefix) and 'Brand' in c]
    brands = pd.Series(dtype=str)
    for bc in brand_cols:
        brands = pd.concat([brands, df_refig_survey_Clean[bc].astype(str)], ignore_index=True)
    brands = brands[brands.notna() & (brands != '') & (brands != 'nan') & (brands != '0')]
    top = brands.value_counts().head(8)
    if len(top) > 0:
        print(f"[{label}]: " + ", ".join(f"{b}({n})" for b,n in top.items()))

=== ยี่ห้อแต่ละประเภท ===
[Open]: Themedez(2068), VSR(1865), Panasonic(1265), Carrier(355), Systemform(234), Sanyo(68), SANYO(48), SANYO SMI(42)
[Bev Walk In]: Themedez(99), Panasonic(92), VSR(54), Systemform(10), Sanyo(8), Carrier(7), SANYO(6), Other(4)
[Ice Cream]: LIEBHERR(361), Liebherr(251), Hiron(228), Other(98), The Cool(83), Haier(65), LIEBHERR (38), Panasonic(29)
[Ice]: Panasonic(860), LIEBHERR(412), Themedez(337), Liebherr(269), Hiron(228), The Cool(211), Other(127), VSR(112)
[Bev Plugin]: Themedez(682), VSR(500), Panasonic(257), Carrier(114), Pattana Inter Cool(97), Systemform(76), Sanden(52), Sanyo(15)
[Bev Remote]: Themedez(233), VSR(169), Panasonic(59), Systemform(23), Carrier(17), Pattana Inter Cool(13), COOLPOINT(7), Dixell(5)
[SS 4D]: Themedez(318), VSR(216), Panasonic(206), Sanden(118), Carrier(72), Systemform(65), Other(23), Sanyo(22)


In [24]:
cols = df_refig_survey_Clean.columns.tolist()
type_prefixes = {
    'Open':       'Ref Open',
    'Bev Walk In':'Ref Bev Walk In',
    'Frozen 3D':  'Ref Frozen3 Door',
    'Ice Cream':  'Ref Ice Cream',
    'Ice':        'Ref Ice',
    'Frozen 1D':  'Ref Frozen1 Door',
    'Bev Plugin': 'Ref Bev Plugin',
    'Frozen 2D':  'Ref Frozen2 Door',
    'Bev Remote': 'Ref Bev Remote',
    'SS 4D':      'Ref Ss4 Door',
}

print("=== สินค้าที่ใส่ในตู้ ===")
for label, prefix in type_prefixes.items():
    prod_cols = [c for c in cols if c.startswith(prefix) and 'Product' in c]
    products = pd.Series(dtype=str)
    for pc in prod_cols:
        products = pd.concat([products, df_refig_survey_Clean[pc].astype(str)], ignore_index=True)
    products = products[products.notna() & (products != '') & (products != 'nan') & (products != '0')]
    top = products.value_counts().head(8)
    if len(top) > 0:
        print(f"[{label}]: " + ", ".join(f"{p}({n})" for p,n in top.items()))
    else:
        print(f"[{label}]: ไม่มีข้อมูล Product")

=== สินค้าที่ใส่ในตู้ ===
[Open]: อื่นๆ(1581), อาหารพร้อมทาน(1100), ยำ/โยเกิร์ต/น้ำผลไม้(985), หมู/ไก่/เนื้อสัตว์(975), แซนวิช/แฮมเบอร์เกอร์/แพนเวิ่ง(591), แซนวิช/แฮมเบอร์เกอร์/แพนเวิ่ง, อาหารพร้อมทาน(289), ยำ/โยเกิร์ต/น้ำผลไม้, อื่นๆ(257), อาหารพร้อมทาน, อื่นๆ(131)
[Bev Walk In]: เครื่องดื่ม(297), อื่นๆ(4), เครื่องดื่ม, อื่นๆ(3), เครื่องดื่ม, อาหารสด(2), อาหารสด(2)
[Frozen 3D]: ไม่มีข้อมูล Product
[Ice Cream]: ไอศครีม(4)
[Ice]: ไอศครีม(4), น้ำแข็ง(3)
[Frozen 1D]: ไม่มีข้อมูล Product
[Bev Plugin]: เครื่องดื่ม(1706), อื่นๆ(81), อาหารสด(40), เครื่องดื่ม, อื่นๆ(18), เครื่องดื่ม, อาหารสด(5), อาหารสด, อื่นๆ(1)
[Frozen 2D]: ไม่มีข้อมูล Product
[Bev Remote]: เครื่องดื่ม(482), อื่นๆ(19), เครื่องดื่ม, อื่นๆ(5), เครื่องดืม(5), อาหารสด(3), เครื่องดื่ม, อาหารสด(3), น้ำอัดลม(2)
[SS 4D]: ไม่มีข้อมูล Product


In [25]:
cols = df_refig_survey_Clean.columns.tolist()
# ตรวจ brand columns สำหรับ Frozen types
for prefix in ['Ref Frozen', 'Ref Ss4', 'Ref Ice Cream']:
    brand_cols = [c for c in cols if c.startswith(prefix) and 'Brand' in c]
    print(f"{prefix}: brand_cols = {brand_cols[:5]}")
    # ตรวจ Asset No, Model, Type columns ด้วย
    other_cols = [c for c in cols if c.startswith(prefix)][:10]
    print(f"  first cols: {other_cols}")

Ref Frozen: brand_cols = ['Ref Frozen Brand_1', 'Ref Frozen Brand_3', 'Ref Frozen Brand_2', 'Ref Frozen Brand_4', 'Ref Frozen Brand_6']
  first cols: ['Ref Frozen3 Door Qty', 'Ref Frozen1 Door Qty', 'Ref Frozen Brand_1', 'Ref Frozen Asset No_1', 'Ref Frozen Asset No_3', 'Ref Frozen Brand_3', 'Ref Frozen Brand_2', 'Ref Frozen Brand_4', 'Ref Frozen Asset No_2', 'Ref Frozen Asset No_4']
Ref Ss4: brand_cols = ['Ref Ss4 Door Brand_2', 'Ref Ss4 Door Brand_1', 'Ref Ss4 Door Brand_3', 'Ref Ss4 Door Brand_4', 'Ref Ss4 Door Brand_5']
  first cols: ['Ref Ss4 Door Brand_2', 'Ref Ss4 Door Asset No_1', 'Ref Ss4 Door Asset No_2', 'Ref Ss4 Door Brand_1', 'Ref Ss4 Door Qty', 'Ref Ss4 Door Status_1', 'Ref Ss4 Door Status_2', 'Ref Ss4 Door Brand_3', 'Ref Ss4 Door Asset No_3', 'Ref Ss4 Door Status_3']
Ref Ice Cream: brand_cols = ['Ref Ice Cream Brand_1', 'Ref Ice Cream Brand_3', 'Ref Ice Cream Brand_2', 'Ref Ice Cream Brand_4']
  first cols: ['Ref Ice Cream Qty', 'Ref Ice Cream Asset No_1', 'Ref Ice Cream

In [26]:
cols = df_refig_survey_Clean.columns.tolist()
# ยี่ห้อ Frozen (รวม 1D, 2D, 3D) และ Ice Cream
for label, prefix in [('Frozen (ทุกขนาด)', 'Ref Frozen'), ('Ice Cream', 'Ref Ice Cream'), ('Ice', 'Ref Ice B')]:
    brand_cols = [c for c in cols if c.startswith(prefix) and 'Brand' in c]
    brands = pd.Series(dtype=str)
    for bc in brand_cols:
        brands = pd.concat([brands, df_refig_survey_Clean[bc].astype(str)], ignore_index=True)
    brands = brands[brands.notna() & (brands != '') & (brands != 'nan') & (brands != '0')]
    top = brands.value_counts().head(10)
    print(f"[{label}]: " + ", ".join(f"{b}({n})" for b,n in top.items()))

[Frozen (ทุกขนาด)]: Themedez(849), Sanden(590), Panasonic(512), VSR(314), Carrier(139), iarp(133), Systemform(110), Iarp(67), IARP(65), Sanyo(46)
[Ice Cream]: LIEBHERR(361), Liebherr(251), Hiron(228), Other(98), The Cool(83), Haier(65), LIEBHERR (38), Panasonic(29), Themedez(27), Sanden(20)
[Ice]: Panasonic(831), Themedez(310), The Cool(128), VSR(105), Sanden(88), MIRAGE(55), LIEBHERR(51), Mirage(36), Carrier(30), Other(29)


In [27]:
import os
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side, numbers
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo
from datetime import datetime

OUTPUT_PATH = os.path.join(os.path.dirname(os.path.abspath("Refig_Survey_Analysys.ipynb")), "Refig_Survey_Report.xlsx")

cols = df_refig_survey_Clean.columns.tolist()

type_prefixes = {
    'Ref Open':       'Ref Open',
    'Bev Walk In':    'Ref Bev Walk In',
    'Frozen 3D':      'Ref Frozen3 Door',
    'Ice Cream':      'Ref Ice Cream',
    'Ice':            'Ref Ice',
    'Frozen 1D':      'Ref Frozen1 Door',
    'Bev Plugin':     'Ref Bev Plugin',
    'Frozen 2D':      'Ref Frozen2 Door',
    'Bev Remote':     'Ref Bev Remote',
    'SS 4D':          'Ref Ss4 Door',
}

qty_cols = [c for c in cols if c.endswith('Qty')]

wb = Workbook()

# ─────────────────────────────────────────────────────────────
# SHEET 1 : Summary Report
# ─────────────────────────────────────────────────────────────
ws_sum = wb.active
ws_sum.title = "Summary"

HEADER_FILL  = PatternFill("solid", fgColor="1F4E79")
HEADER_FONT  = Font(bold=True, color="FFFFFF", size=11)
SUBHDR_FILL  = PatternFill("solid", fgColor="2E75B6")
SUBHDR_FONT  = Font(bold=True, color="FFFFFF", size=10)
ALT_FILL     = PatternFill("solid", fgColor="D6E4F0")
TOTAL_FILL   = PatternFill("solid", fgColor="BDD7EE")
TOTAL_FONT   = Font(bold=True, size=10)
BORDER       = Border(
    left   = Side(style='thin'),
    right  = Side(style='thin'),
    top    = Side(style='thin'),
    bottom = Side(style='thin'),
)

def set_cell(ws, row, col, value, fill=None, font=None, align_center=False, number_format=None):
    cell = ws.cell(row=row, column=col, value=value)
    cell.border = BORDER
    if fill:   cell.fill = fill
    if font:   cell.font = font
    if align_center: cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    if number_format: cell.number_format = number_format
    return cell

# ── Title ──
ws_sum.merge_cells("A1:F1")
t = ws_sum["A1"]
t.value = f"สรุปข้อมูลตู้แช่ (Refrigeration Survey)  —  Export: {datetime.today().strftime('%d/%m/%Y')}"
t.font = Font(bold=True, size=14, color="FFFFFF")
t.fill = HEADER_FILL
t.alignment = Alignment(horizontal='center', vertical='center')
ws_sum.row_dimensions[1].height = 30

# ── Section A : จำนวนตู้แต่ละประเภท ──
ws_sum.merge_cells("A2:F2")
c2 = ws_sum["A2"]
c2.value = "A.  จำนวนตู้แช่แต่ละประเภท"
c2.font = SUBHDR_FONT
c2.fill = SUBHDR_FILL
c2.alignment = Alignment(horizontal='left', vertical='center')
ws_sum.row_dimensions[2].height = 20

headers_A = ["ประเภทตู้แช่", "รหัสคอลัมน์", "จำนวน (ตัว)", "จำนวนสาขา", "เฉลี่ย/สาขา"]
for ci, h in enumerate(headers_A, 1):
    set_cell(ws_sum, 3, ci, h, fill=PatternFill("solid", fgColor="2E75B6"),
             font=Font(bold=True, color="FFFFFF", size=10), align_center=True)

label_map = {
    'Ref Open Qty':          'ตู้แช่เปิด (Open)',
    'Ref Bev Walk In Qty':   'ห้องเย็นเครื่องดื่ม (Walk-in)',
    'Ref Frozen3 Door Qty':  'ตู้แช่แข็ง 3 ประตู (Frozen 3D)',
    'Ref Ice Cream Qty':     'ตู้ไอศกรีม (Ice Cream)',
    'Ref Ice Qty':           'ตู้น้ำแข็ง (Ice)',
    'Ref Frozen1 Door Qty':  'ตู้แช่แข็ง 1 ประตู (Frozen 1D)',
    'Ref Bev Plugin Qty':    'ตู้แช่เครื่องดื่ม Plugin (Bev Plugin)',
    'Ref Frozen2 Door Qty':  'ตู้แช่แข็ง 2 ประตู (Frozen 2D)',
    'Ref Bev Remote Qty':    'ตู้แช่เครื่องดื่ม Remote (Bev Remote)',
    'Ref Ss4 Door Qty':      'ตู้ SS 4 ประตู (SS 4D)',
}

row = 4
grand_total_units = 0
for i, qc in enumerate(qty_cols):
    nums = pd.to_numeric(df_refig_survey_Clean[qc], errors='coerce')
    total_u = int(nums.sum())
    store_c = int((nums > 0).sum())
    avg = round(total_u / store_c, 2) if store_c > 0 else 0
    grand_total_units += total_u
    fill = ALT_FILL if i % 2 == 0 else None
    set_cell(ws_sum, row, 1, label_map.get(qc, qc), fill=fill)
    set_cell(ws_sum, row, 2, qc, fill=fill)
    set_cell(ws_sum, row, 3, total_u, fill=fill, align_center=True, number_format='#,##0')
    set_cell(ws_sum, row, 4, store_c, fill=fill, align_center=True, number_format='#,##0')
    set_cell(ws_sum, row, 5, avg, fill=fill, align_center=True, number_format='#,##0.00')
    row += 1

# Grand total row
set_cell(ws_sum, row, 1, "รวมทั้งหมด", fill=TOTAL_FILL, font=TOTAL_FONT)
set_cell(ws_sum, row, 2, "", fill=TOTAL_FILL)
set_cell(ws_sum, row, 3, grand_total_units, fill=TOTAL_FILL, font=TOTAL_FONT, align_center=True, number_format='#,##0')
set_cell(ws_sum, row, 4, "", fill=TOTAL_FILL)
set_cell(ws_sum, row, 5, "", fill=TOTAL_FILL)
row += 2

# ── Section B : ยี่ห้อ (Brand) ──
ws_sum.merge_cells(f"A{row}:F{row}")
ws_sum.cell(row=row, column=1).value = "B.  ยี่ห้อตู้แช่แต่ละประเภท (Top 8)"
ws_sum.cell(row=row, column=1).font = SUBHDR_FONT
ws_sum.cell(row=row, column=1).fill = SUBHDR_FILL
ws_sum.cell(row=row, column=1).alignment = Alignment(horizontal='left', vertical='center')
ws_sum.row_dimensions[row].height = 20
row += 1

headers_B = ["ประเภทตู้แช่", "อันดับ", "ยี่ห้อ", "จำนวน (ครั้ง)"]
for ci, h in enumerate(headers_B, 1):
    set_cell(ws_sum, row, ci, h, fill=PatternFill("solid", fgColor="2E75B6"),
             font=Font(bold=True, color="FFFFFF", size=10), align_center=True)
row += 1

brand_prefixes = {
    'ตู้แช่เปิด':              ('Ref Open',       'Ref Open'),
    'ห้องเย็นเครื่องดื่ม':     ('Bev Walk In',    'Ref Bev Walk In'),
    'ตู้แช่แข็ง (1D/2D/3D)':  ('Frozen',         'Ref Frozen'),
    'ตู้ไอศกรีม':              ('Ice Cream',      'Ref Ice Cream'),
    'ตู้น้ำแข็ง':              ('Ice',            'Ref Ice B'),
    'ตู้แช่เครื่องดื่ม Plugin': ('Bev Plugin',    'Ref Bev Plugin'),
    'ตู้แช่เครื่องดื่ม Remote': ('Bev Remote',    'Ref Bev Remote'),
    'ตู้ SS 4 ประตู':           ('SS 4D',         'Ref Ss4 Door'),
}

alt = 0
for type_label, (_, prefix) in brand_prefixes.items():
    brand_cols_t = [c for c in cols if c.startswith(prefix) and 'Brand' in c]
    brands_s = pd.Series(dtype=str)
    for bc_col in brand_cols_t:
        brands_s = pd.concat([brands_s, df_refig_survey_Clean[bc_col].astype(str)], ignore_index=True)
    brands_s = brands_s[brands_s.notna() & (brands_s != '') & (brands_s != 'nan') & (brands_s != '0')]
    top_brands = brands_s.value_counts().head(8)
    if len(top_brands) == 0:
        continue
    first = True
    for rank, (brand_name, cnt_b) in enumerate(top_brands.items(), 1):
        fill = ALT_FILL if alt % 2 == 0 else None
        set_cell(ws_sum, row, 1, type_label if first else "", fill=fill)
        set_cell(ws_sum, row, 2, rank, fill=fill, align_center=True)
        set_cell(ws_sum, row, 3, brand_name, fill=fill)
        set_cell(ws_sum, row, 4, int(cnt_b), fill=fill, align_center=True, number_format='#,##0')
        first = False
        row += 1
    alt += 1

# ── Section C : สินค้า (Product) ──
row += 1
ws_sum.merge_cells(f"A{row}:F{row}")
ws_sum.cell(row=row, column=1).value = "C.  สินค้าในตู้แต่ละประเภท (Top 8)"
ws_sum.cell(row=row, column=1).font = SUBHDR_FONT
ws_sum.cell(row=row, column=1).fill = SUBHDR_FILL
ws_sum.cell(row=row, column=1).alignment = Alignment(horizontal='left', vertical='center')
ws_sum.row_dimensions[row].height = 20
row += 1

headers_C = ["ประเภทตู้แช่", "อันดับ", "สินค้า", "จำนวน (ครั้ง)"]
for ci, h in enumerate(headers_C, 1):
    set_cell(ws_sum, row, ci, h, fill=PatternFill("solid", fgColor="2E75B6"),
             font=Font(bold=True, color="FFFFFF", size=10), align_center=True)
row += 1

product_prefixes = {
    'ตู้แช่เปิด':               'Ref Open',
    'ห้องเย็นเครื่องดื่ม':      'Ref Bev Walk In',
    'ตู้ไอศกรีม':               'Ref Ice Cream',
    'ตู้น้ำแข็ง':               'Ref Ice',
    'ตู้แช่เครื่องดื่ม Plugin':  'Ref Bev Plugin',
    'ตู้แช่เครื่องดื่ม Remote':  'Ref Bev Remote',
}

alt2 = 0
for type_label, prefix in product_prefixes.items():
    prod_cols_t = [c for c in cols if c.startswith(prefix) and 'Product' in c]
    products_s = pd.Series(dtype=str)
    for pc_col in prod_cols_t:
        products_s = pd.concat([products_s, df_refig_survey_Clean[pc_col].astype(str)], ignore_index=True)
    products_s = products_s[products_s.notna() & (products_s != '') & (products_s != 'nan') & (products_s != '0')]
    top_prods = products_s.value_counts().head(8)
    if len(top_prods) == 0:
        continue
    first = True
    for rank, (prod_name, cnt_p) in enumerate(top_prods.items(), 1):
        fill = ALT_FILL if alt2 % 2 == 0 else None
        set_cell(ws_sum, row, 1, type_label if first else "", fill=fill)
        set_cell(ws_sum, row, 2, rank, fill=fill, align_center=True)
        set_cell(ws_sum, row, 3, prod_name, fill=fill)
        set_cell(ws_sum, row, 4, int(cnt_p), fill=fill, align_center=True, number_format='#,##0')
        first = False
        row += 1
    alt2 += 1

# column widths
for col_idx, w in zip(range(1, 7), [40, 30, 18, 14, 14, 14]):
    ws_sum.column_dimensions[get_column_letter(col_idx)].width = w

# ─────────────────────────────────────────────────────────────
# SHEET 2 : Raw Data (Refig Survey Clean)
# ─────────────────────────────────────────────────────────────
ws_raw = wb.create_sheet("Raw Data - Refig")
raw_df = df_refig_survey_Clean.copy()

# Write header
for ci, col_name in enumerate(raw_df.columns, 1):
    cell = ws_raw.cell(row=1, column=ci, value=col_name)
    cell.fill = HEADER_FILL
    cell.font = HEADER_FONT
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    cell.border = BORDER

# Write data
for ri, data_row in enumerate(raw_df.itertuples(index=False), 2):
    fill = ALT_FILL if ri % 2 == 0 else None
    for ci, val in enumerate(data_row, 1):
        cell = ws_raw.cell(row=ri, column=ci, value=val if val != '' else None)
        cell.border = BORDER
        if fill: cell.fill = fill
        cell.alignment = Alignment(vertical='center')

# Excel Table on Raw sheet
table_ref = f"A1:{get_column_letter(len(raw_df.columns))}{len(raw_df)+1}"
tbl = Table(displayName="RefigSurveyRaw", ref=table_ref)
tbl.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showFirstColumn=False,
                                    showLastColumn=False, showRowStripes=True, showColumnStripes=False)
ws_raw.add_table(tbl)

# auto-width first 15 cols only (others may be too many)
for ci in range(1, min(16, len(raw_df.columns)+1)):
    col_vals = [str(raw_df.iloc[r, ci-1]) for r in range(min(200, len(raw_df)))]
    max_len = max([len(str(raw_df.columns[ci-1]))] + [len(v) for v in col_vals if v not in ('', 'nan', 'None')])
    ws_raw.column_dimensions[get_column_letter(ci)].width = min(max_len + 2, 40)

ws_raw.freeze_panes = "A2"

# ─────────────────────────────────────────────────────────────
# SHEET 3 : Raw Data (Merged)
# ─────────────────────────────────────────────────────────────
ws_merged = wb.create_sheet("Raw Data - Merged")
mg_df = merged_critical_df.copy()

for ci, col_name in enumerate(mg_df.columns, 1):
    cell = ws_merged.cell(row=1, column=ci, value=col_name)
    cell.fill = HEADER_FILL
    cell.font = HEADER_FONT
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    cell.border = BORDER

for ri, data_row in enumerate(mg_df.itertuples(index=False), 2):
    fill = ALT_FILL if ri % 2 == 0 else None
    for ci, val in enumerate(data_row, 1):
        cell = ws_merged.cell(row=ri, column=ci, value=val if val != '' else None)
        cell.border = BORDER
        if fill: cell.fill = fill
        cell.alignment = Alignment(vertical='center')

table_ref2 = f"A1:{get_column_letter(len(mg_df.columns))}{len(mg_df)+1}"
tbl2 = Table(displayName="MergedSurveyRaw", ref=table_ref2)
tbl2.tableStyleInfo = TableStyleInfo(name="TableStyleMedium6", showFirstColumn=False,
                                     showLastColumn=False, showRowStripes=True, showColumnStripes=False)
ws_merged.add_table(tbl2)
ws_merged.freeze_panes = "A2"

for ci in range(1, min(16, len(mg_df.columns)+1)):
    col_vals = [str(mg_df.iloc[r, ci-1]) for r in range(min(200, len(mg_df)))]
    max_len = max([len(str(mg_df.columns[ci-1]))] + [len(v) for v in col_vals if v not in ('', 'nan', 'None')])
    ws_merged.column_dimensions[get_column_letter(ci)].width = min(max_len + 2, 40)

# ─────────────────────────────────────────────────────────────
# Save
# ─────────────────────────────────────────────────────────────
wb.save(OUTPUT_PATH)
print(f"✅ Saved: {OUTPUT_PATH}")
print(f"   Sheet 'Summary'          : {row-1} rows")
print(f"   Sheet 'Raw Data - Refig' : {len(raw_df)} rows × {len(raw_df.columns)} cols")
print(f"   Sheet 'Raw Data - Merged': {len(mg_df)} rows × {len(mg_df.columns)} cols")

✅ Saved: g:\My Drive\00. Web Projects\Air_Ref_Servey\Pyhon ANL\Refig_Survey_Report.xlsx
   Sheet 'Summary'          : 113 rows
   Sheet 'Raw Data - Refig' : 1260 rows × 888 cols
   Sheet 'Raw Data - Merged': 1396 rows × 1178 cols
